# Fase 2: Creación de la Base de Conocimiento RAG---En este notebook construiremos la **base de datos vectorial** que usará el agente de IA para responder preguntas frecuentes de clientes.### ¿Qué es RAG?**RAG (Retrieval-Augmented Generation)** es una técnica que permite a un modelo de lenguaje (LLM) buscar información relevante en una base de datos propia *antes* de responder. Así el agente:- ✅ Responde con datos reales de la empresa (no inventa)- ✅ Puede actualizarse sin reentrenar el modelo- ✅ Funciona 100% en local (los datos no salen del ordenador)### Flujo de esta fase:```Carga  →  Fragmentación  →  Embeddings  →  ChromaDB(loader)   (chunking)       (vectores)     (BBDD local)```---### 📋 Requisitos previos- Python 3.10+- Entorno virtual activo (`.venv`)- Conexión a internet (solo para descargar el modelo de embeddings la primera vez)

---## Paso 1 — Instalación de libreríasInstalamos todos los paquetes necesarios. Solo hay que ejecutar esto **una vez** por entorno.| Librería | Para qué sirve ||---|---|| `langchain` | Framework orquestador del agente || `langchain-chroma` | Conector con ChromaDB || `langchain-huggingface` | Conector con modelos de HuggingFace || `langchain-texto-splitters` | Herramienta para fragmentar texto || `sentence-transformers` | Motor de embeddings local (sin GPU) |

In [ ]:
# Instalamos las librerías del ecosistema LangChain + ChromaDB + HuggingFace%pip install langchain langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers langchain-community

---## Paso 2 — Cargar el documento con LangChainUsamos `TextLoader` para leer el archivo `faqs.txt` y convertirlo en un objeto `Document` que LangChain puede procesar.> LangChain tiene loaders para PDF, Word, páginas web, YouTube, bases de datos, etc.> **Importante:** El archivo se encontrará en la misma carpeta que este notebook.

In [ ]:
from langchain_community.document_loaders import TextLoaderprint("Cargando el documento de FAQs...")# Usamos utf-8 para no tener problemas con acentos y caracteres especiales en españolloader = TextLoader("faqs.txt", encoding="utf-8")documentos = loader.load()print(f"   Documento cargado. Número de documentos: {len(documentos)}")print(f"   Primeros 200 caracteres del documento cargado:")print(f"   '{documentos[0].page_content[:200]}...'")

---## Paso 3 — Fragmentar el texto (Chunking)Dividimos el documento en **fragmentos pequeños (chunks)**. Esto es crucial para que la búsqueda sea precisa.### ¿Por qué fragmentar?Imagina que el agente busca información sobre devoluciones. Si pasáramos todo el documento, habría mucho "ruido". Fragmentando, recuperamos solo los párrafos exactos que hablan de devoluciones.### Parámetros importantes:- **`chunk_size=300`**: Cada fragmento tendrá ~300 caracteres (una idea completa)- **`chunk_overlap=60`**: Los fragmentos se solapan 60 caracteres para no perder contexto entre trozos

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitterprint("Fragmentando el texto en chunks...")text_splitter = RecursiveCharacterTextSplitter(    chunk_size=300,    # Cada fragmento tendrá ~300 caracteres    chunk_overlap=60,  # Solapamos 60 caracteres para no perder contexto    length_function=len,    separators=["\n\n", "\n", ". ", " ", ""]  # Prioridad de separación)chunks = text_splitter.split_documents(documentos)print(f"Documento dividido en {len(chunks)} fragmentos.")print()# Mostramos todos los fragmentos para entender cómo quedaronfor i, chunk in enumerate(chunks):    print(f"--- Fragmento {i+1} ({len(chunk.page_content)} chars) ---")    print(chunk.page_content)    print()

---## Paso 4 — Cargar el modelo de Embeddings (Vectorización del texto)Los **embeddings** convierten el texto en vectores numéricos. Textos con significados similares producirán vectores parecidos, lo que permite la búsqueda semántica.### Modelo elegido: `paraphrase-multilingual-MiniLM-L12-v2`- ✅ **Multilingüe**: Funciona perfectamente en español- ✅ **Ligero**: No necesita GPU potente (~130MB)- ✅ **Local**: Una vez descargado, no necesita internet- ✅ **Preciso**: Entiende sinónimos y variaciones del lenguaje> **Primera ejecución:** el modelo se descargará automáticamente (~130 MB). Las siguientes ejecuciones serán instantáneas.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddingsprint("  Cargando el modelo de Embeddings (puede tardar en la primera ejecución)...")print("  Modelo: paraphrase-multilingual-MiniLM-L12-v2")print()embeddings_model = HuggingFaceEmbeddings(    model_name="paraphrase-multilingual-MiniLM-L12-v2")print(" Modelo de embeddings cargado correctamente.")# Demostramos cómo funciona un embeddingtexto_prueba = "¿Cuánto tarda el envío?"vector_prueba = embeddings_model.embed_query(texto_prueba)print()print(f" Ejemplo: La frase '{texto_prueba}'")print(f" se convierte en un vector de {len(vector_prueba)} dimensiones.")print(f" Primeros 5 valores: {vector_prueba[:5]}")

---## Paso 5 — Crear la Base de Datos Vectorial con ChromaDB**ChromaDB** es nuestra base de datos vectorial. Almacena todos los fragmentos del documento junto con sus vectores, y permite hacer búsquedas semánticas ultra-rápidas.### ¿Qué hace `from_documents`?1. Toma cada `chunk` de texto2. Le aplica el modelo de embeddings para obtener su vector3. Guarda el par (texto + vector) en la base de datos4. Persiste todo en la carpeta `./chroma_db` del disco> ⚠️ Si ya existe una base de datos (segunda ejecución), la celda de limpieza de abajo borrará la anterior para crearla de nuevo.

In [ ]:
import shutilimport osfrom langchain_chroma import Chroma# Directorio donde se persistirá la base de datosDIRECTORIO_DB = "./chroma_db"# Si ya existe una base de datos previa, la eliminamos para empezar limpioif os.path.exists(DIRECTORIO_DB):    shutil.rmtree(DIRECTORIO_DB)    print(f"  Base de datos anterior eliminada (para regenerarla limpia).")print(f"")print(f"  Creando base de datos y vectorizando {len(chunks)} fragmentos...")print(f"   (Esto puede tardar 1-2 minutos dependiendo del hardware)")# Creamos la base de datos vectorial Chroma con todos los fragmentosvectorstore = Chroma.from_documents(    documents=chunks,              # Los fragmentos de texto    embedding=embeddings_model,    # El modelo que los convierte en vectores    persist_directory=DIRECTORIO_DB  # La carpeta donde se guardará en disco)print(f"")print(f" ¡Base de conocimiento creada y guardada en la carpeta '{DIRECTORIO_DB}'!")print(f"  Fragmentos indexados: {vectorstore._collection.count()}")# Verificamos que la carpeta existe en discoif os.path.exists(DIRECTORIO_DB):    archivos = os.listdir(DIRECTORIO_DB)    print(f"   Archivos en disco: {archivos}")


⚙️  Creando base de datos y vectorizando 13 fragmentos...
    (Esto puede tardar 1-2 minutos dependiendo del hardware)


---## Paso 6 — Verificación y Prueba de la Base de Datos¡La base de datos está lista! Vamos a probarla haciendo **búsquedas semánticas** para verificar que funciona correctamente.Fíjate que las búsquedas funcionan aunque las palabras no coincidan exactamente con el texto del documento. Eso es el poder de los embeddings.

In [ ]:
# Configuramos el recuperador (retriever) para buscar los 2 fragmentos más relevantesretriever = vectorstore.as_retriever(search_kwargs={"k": 2})# Lista de preguntas de prueba (simulando correos de clientes)preguntas_prueba = [    "¿Cuánto tiempo tarda en llegar mi pedido?",    "Quiero devolver un producto que me llegó roto",    "¿Puedo pagar con PayPal?",    "¿Cuál es vuestro horario de atención?",    "Compré un router en Amazon y no funciona, ¿me ayudáis?"]print("🔍 PRUEBA DE BÚSQUEDA SEMÁNTICA EN LA BASE DE CONOCIMIENTO")print("=" * 60)for pregunta in preguntas_prueba:    print(f"\n Pregunta: '{pregunta}'")    print(f"   Fragmentos recuperados:")        resultados = retriever.invoke(pregunta)        for j, doc in enumerate(resultados):        print(f"   [{j+1}] → {doc.page_content.strip()[:150]}...")        print("-" * 60)

---## 📊 Resumen de la Fase 2| Paso | Acción | Resultado ||------|--------|----------|| 1 | Instalación de librerías | Entorno listo || 2 | Carga del documento `faqs.txt` | 1 documento LangChain || 3 | Fragmentación (chunking) | ~15 fragmentos de ~300 chars || 4 | Modelo de embeddings | `MiniLM-L12-v2` multilingüe cargado || 5 | Creación de ChromaDB | Carpeta `./chroma_db` en disco || 6 | Verificación | Búsquedas semánticas correctas |